# Multi-Turn Evaluation & Conversation Simulation with Amazon SageMaker AI MLflow Apps

This notebook accompanies the AWS blog post *Evaluate conversational AI agents with multi-turn evaluation in serverless MLflow on Amazon SageMaker AI*.

It demonstrates:

1. A traced conversational agent built with the **Strands Agents SDK** running against a **SageMaker AI inference endpoint**, with automatic tracing via `mlflow.strands.autolog()`
2. Session-ID tagging to group turns into conversations
3. **Version tracking** using `mlflow.set_active_model()` so every trace and evaluation result links to a specific agent version
4. Evaluation of pre-recorded conversations with built-in multi-turn scorers (`ConversationCompleteness`, `UserFrustration`, `KnowledgeRetention`)
5. A custom multi-turn judge using `make_judge` with the `{{ conversation }}` template variable
6. Conversation simulation with `ConversationSimulator` + `predict_fn` for regression testing
7. A v1-vs-v2 comparison using version tracking and the simulator together

## Prerequisites

- An Amazon SageMaker AI MLflow Apps [MLflow App](https://aws.amazon.com/blogs/aws/accelerate-ai-development-using-amazon-sagemaker-ai-with-serverless-mlflow/)
- A SageMaker AI real-time inference endpoint serving **Qwen3.5-0.8B** with an OpenAI-compatible chat completions API — deploy it with the companion script [`deploy_qwen_sagemaker.py`](./deploy_qwen_sagemaker.py) (see the README)
- Access to Anthropic Claude Sonnet 4.6 on Amazon Bedrock (used as the judge model)
- AWS credentials with `sagemaker:InvokeEndpoint` and `bedrock:InvokeModel` permissions
- Python 3.12+


## 0. Install Dependencies


In [ ]:
%pip install --quiet "mlflow==3.11.1" sagemaker-mlflow "strands-agents[sagemaker]" strands-agents-tools boto3


## 1. Configuration

Fill in your MLflow App ARN, SageMaker AI endpoint name, and AWS region. The judge model is a Bedrock-hosted Claude Sonnet 4.6.


In [ ]:
REGION = "us-east-1"

# SageMaker AI endpoint hosting the chat model (OpenAI-compatible chat completions API).
# Created by the companion script: python deploy_qwen_sagemaker.py
ENDPOINT_NAME = "qwen3-5-0-8b"  # Replace if you deployed under a different name

# MLflow App ARN — created from the SageMaker Studio MLflow UI
TRACKING_ARN = "arn:aws:sagemaker:us-east-1:<account-id>:mlflow-app/<app-id>"  # Replace
EXPERIMENT_NAME = "multi-turn-eval-demo"

# Judge model — native MLflow support for Bedrock; no extra setup needed
JUDGE_MODEL = "bedrock:/global.anthropic.claude-sonnet-4-6"

SYSTEM_PROMPT = (
    "You are a helpful ML engineering assistant. "
    "Answer questions about MLflow, SageMaker, and machine learning best practices. "
    "Be concise and accurate."
)


## 2. Connect to the MLflow App and enable Strands auto-tracing


In [ ]:
import mlflow

mlflow.set_tracking_uri(TRACKING_ARN)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.strands.autolog()

print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment   : {experiment.name} (id={experiment.experiment_id})")


Because the built-in judges and `make_judge` call Bedrock directly (via the `bedrock:/` URI), we make sure AWS credentials are available as environment variables for MLflow's judge process.


In [ ]:
import os
import boto3

credentials = boto3.Session().get_credentials().get_frozen_credentials()
os.environ["AWS_ACCESS_KEY_ID"] = credentials.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = credentials.secret_key
if credentials.token:
    os.environ["AWS_SESSION_TOKEN"] = credentials.token
os.environ["AWS_REGION"] = REGION


## 3. Version-track your agent

MLflow's [`LoggedModel`](https://mlflow.org/docs/latest/genai/version-tracking/) captures a snapshot of your agent — code reference, config, and associated traces and evaluation results — under a single version identifier. Calling `mlflow.set_active_model(name=version_name)` makes every subsequent trace **and** every evaluation result link to that version, which is what makes v1-vs-v2 comparisons possible later in the notebook.

We derive the version name from the current Git commit when available, and fall back to a timestamped local identifier otherwise.


In [ ]:
from datetime import datetime, timezone
from mlflow.utils.git_utils import get_git_commit

APP_NAME = "strands-multi-turn-eval-agent"

git_commit = get_git_commit(".")
if git_commit:
    version_suffix = f"git-{git_commit[:8]}"
else:
    version_suffix = f"local-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

VERSION_V1 = f"{APP_NAME}-v1-{version_suffix}"

# Every trace and evaluation result from this point on links to VERSION_V1
mlflow.set_active_model(name=VERSION_V1)
print(f"Active model version: {VERSION_V1}")


## 4. Define the Conversational Agent

A Strands `Agent` backed by `SageMakerAIModel`. Each turn is wrapped in an `@mlflow.trace` function that attaches a session ID so MLflow can group turns into conversations.


In [ ]:
from strands import Agent
from strands.models.sagemaker import SageMakerAIModel

# Qwen sampling presets from the Qwen model card (valid for Qwen3.5 and 3.6).
# https://huggingface.co/Qwen/Qwen3.5-0.8B#best-practices
#
# The SageMaker AI endpoint (created by deploy_qwen_sagemaker.py) is
# launched with vLLM's --reasoning-parser qwen3,
# so <think>...</think> blocks are stripped server-side into a separate
# reasoning field. The assistant message the judge sees is always clean.
#
# Strands' SageMakerAIModel only recognizes a fixed list of payload keys
# (max_tokens, stream, temperature, top_p, top_k, stop,
#  tool_results_as_user_messages, additional_args). Non-standard keys must
# go under `additional_args`, which Strands merges into the request body
# alongside the recognized ones. See github.com/strands-agents/sdk-python/issues/815

# Instruct (non-thinking) preset — recommended for conversational assistants
# being graded on conversation-level quality.
QWEN_INSTRUCT_PAYLOAD = {
    "max_tokens": 4096,
    "stream": True,
    "temperature": 0.7,
    "top_p": 0.80,
    "top_k": 20,
    "additional_args": {
        "min_p": 0.0,
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0,
        "chat_template_kwargs": {"enable_thinking": False},
    },
}

# Thinking-mode preset (general tasks). Safe to use here because the endpoint
# has server-side reasoning parsing enabled.
QWEN_THINKING_PAYLOAD = {
    "max_tokens": 8192,
    "stream": True,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 20,
    "additional_args": {
        "min_p": 0.0,
        "presence_penalty": 0.0,
        "repetition_penalty": 1.0,
        "chat_template_kwargs": {"enable_thinking": True, "preserve_thinking": True},
    },
}


def build_agent(
    system_prompt: str = SYSTEM_PROMPT,
    payload_config: dict | None = None,
    endpoint_name: str = ENDPOINT_NAME,
) -> Agent:
    """Create a fresh Strands agent bound to the given SageMaker AI endpoint."""
    model = SageMakerAIModel(
        endpoint_config={"endpoint_name": endpoint_name, "region_name": REGION},
        payload_config=payload_config or QWEN_INSTRUCT_PAYLOAD,
    )
    # callback_handler=None silences Strands' default stdout printing,
    # so only our run_session prints appear in the notebook output.
    # MLflow tracing still captures everything via mlflow.strands.autolog().
    return Agent(
        model=model,
        system_prompt=system_prompt,
        callback_handler=None,
    )


def _extract_text(message) -> str:
    """Extract plain text from a Strands AgentResult.message (dict or string)."""
    if isinstance(message, str):
        return message
    if isinstance(message, dict):
        content = message.get("content", [])
        if isinstance(content, list):
            return "".join(c.get("text", "") for c in content if isinstance(c, dict))
        if isinstance(content, str):
            return content
    return str(message)


@mlflow.trace
def chat_turn(agent: Agent, user_message: str, session_id: str) -> str:
    """One conversational turn. The trace is tagged with the session ID for multi-turn grouping."""
    mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
    result = agent(user_message)
    return _extract_text(result.message)


## 5. Generate Pre-Recorded Conversations

We run two contrasting sessions — one smooth, one where the user becomes frustrated — to populate the experiment with traces. A single `Agent` instance keeps history across turns inside one session.


In [ ]:
import uuid


def run_session(turns: list[str], agent_factory=build_agent) -> str:
    session_id = f"session-{uuid.uuid4().hex[:8]}"
    agent = agent_factory()
    print(f"=== Session {session_id} ===")
    for user_msg in turns:
        reply = chat_turn(agent, user_msg, session_id=session_id)
        print(f"  User : {user_msg}")
        print(f"  Agent: {reply}\n")
    return session_id


smooth = run_session([
    "What is MLflow experiment tracking?",
    "How do I log parameters and metrics in a run?",
    "Can I compare runs across experiments?",
])

frustrated = run_session([
    "I'm deploying a model to SageMaker but getting an error.",
    "I already told you it's a SageMaker deployment error. Can you help or not?",
    "Fine, what about using MLflow with SageMaker instead?",
])


## 6. Evaluate Pre-Generated Conversations with Session-Level Scorers

Retrieve the traces and pass them to `mlflow.genai.evaluate`. MLflow groups traces by the `mlflow.trace.session` metadata tag automatically.

| Built-in judge | What it evaluates |
|---|---|
| `ConversationCompleteness` | Were all user questions addressed by the end? |
| `UserFrustration` | Did the user become frustrated? Was it resolved? |
| `KnowledgeRetention` | Does the agent remember info from earlier turns? |


In [ ]:
from mlflow.genai.scorers import (
    ConversationCompleteness,
    UserFrustration,
    KnowledgeRetention,
)

# Retrieve only the traces of the two sessions generated above. An unfiltered
# search would return every trace ever logged to the experiment — including
# ones from previous notebook runs — and each extra session multiplies the
# number of judge (Bedrock) calls, making this cell slower on every re-run.
traces = []
for session_id in [smooth, frustrated]:
    traces += mlflow.search_traces(
        locations=[experiment.experiment_id],
        filter_string=f"metadata.`mlflow.trace.session` = '{session_id}'",
        return_type="list",
    )
print(f"Retrieved {len(traces)} traces across 2 sessions")

results = mlflow.genai.evaluate(
    data=traces,
    scorers=[
        ConversationCompleteness(model=JUDGE_MODEL),
        UserFrustration(model=JUDGE_MODEL),
        KnowledgeRetention(model=JUDGE_MODEL),
    ],
)

print("\n=== Session-Level Metrics ===")
for metric, value in results.metrics.items():
    print(f"  {metric}: {value}")


Multi-turn assessments are stored on the **first trace** of each session, so only session-start rows carry scorer values. To see one row per session, filter to the rows with non-empty assessments:


In [ ]:
session_df = results.result_df[results.result_df["assessments"].apply(lambda a: len(a) > 0)]
session_df[[
    "trace_id",
    "conversation_completeness/value",
    "user_frustration/value",
    "knowledge_retention/value",
]]


### Evaluate a specific session

To evaluate a single session, filter `mlflow.search_traces` by the session-ID metadata.


In [ ]:
session_traces = mlflow.search_traces(
    locations=[experiment.experiment_id],
    filter_string=f"metadata.`mlflow.trace.session` = '{frustrated}'",
    return_type="list",
)

single_session_results = mlflow.genai.evaluate(
    data=session_traces,
    scorers=[ConversationCompleteness(model=JUDGE_MODEL), UserFrustration(model=JUDGE_MODEL)],
)
single_session_results.metrics


## 7. Custom Multi-Turn Judge

Use `make_judge` with the `{{ conversation }}` template variable to create domain-specific multi-turn judges. The variable injects the full conversation history into the prompt.


In [ ]:
from typing import Literal
from mlflow.genai.judges import make_judge

tone_consistency_judge = make_judge(
    name="tone_consistency",
    instructions=(
        "Analyze the {{ conversation }} and evaluate whether the assistant "
        "maintains a consistent, professional, and helpful tone throughout "
        "all turns — even when the user is frustrated or unclear. "
        "Rate as 'consistent', 'mostly_consistent', or 'inconsistent'."
    ),
    feedback_value_type=Literal["consistent", "mostly_consistent", "inconsistent"],
    model=JUDGE_MODEL,
)

custom_results = mlflow.genai.evaluate(
    data=traces,
    scorers=[tone_consistency_judge],
)

print("=== Tone Consistency Results ===")
for metric, value in custom_results.metrics.items():
    print(f"  {metric}: {value}")


# Shared scorer list — reused in every multi-turn evaluation below so each
# agent version is graded on exactly the same metrics. This is what makes
# the v1 / v2 / v3 side-by-side comparison meaningful.
MULTI_TURN_SCORERS = [
    ConversationCompleteness(model=JUDGE_MODEL),
    UserFrustration(model=JUDGE_MODEL),
    KnowledgeRetention(model=JUDGE_MODEL),
    tone_consistency_judge,
]


## 8. Conversation Simulation with `ConversationSimulator`

Instead of manually crafting conversations, define test scenarios as goals — optionally with personas and simulation guidelines — and MLflow drives a simulated user against your agent.

Use cases:
- Testing a new agent version with consistent scenarios
- Red-teaming with adversarial personas
- Generating diverse edge cases at scale


In [ ]:
from mlflow.genai.simulators import ConversationSimulator
from mlflow.genai.scorers import ConversationCompleteness, UserFrustration

test_cases = [
    {
        "goal": (
            "Learn how to set up MLflow experiment tracking with SageMaker. "
            "The agent should provide code examples and explain the key concepts."
        ),
    },
    {
        "goal": "Debug why model artifacts are not being logged to the tracking server.",
        "persona": "You are a frustrated data scientist who has been stuck on this for hours.",
    },
    {
        "goal": "Understand the difference between MLflow experiments, runs, and registered models.",
        "persona": "You are a beginner who needs step-by-step explanations.",
        "simulation_guidelines": [
            "Start with a broad question before asking about specifics.",
            "Do not mention registered models until the assistant brings them up.",
        ],
    },
]

simulator = ConversationSimulator(
    test_cases=test_cases,
    max_turns=4,
    user_model=JUDGE_MODEL,
)


### Define the `predict_fn` for the simulator

The simulator calls `predict_fn` at each turn with the full conversation history (as a list of `{role, content}` dicts) plus an `mlflow_session_id` kwarg.

We keep one Strands `Agent` per simulated session (cached on the function object), so each session's conversation manager preserves its own history without cross-session leakage. Each turn goes through the same `chat_turn` wrapper used for pre-recorded conversations, keeping session tagging consistent.


In [ ]:
def make_predict_fn(agent_factory):
    """Build a predict_fn bound to a given agent factory so we can compare versions later."""
    agents: dict[str, Agent] = {}

    def predict_fn(input: list[dict], **kwargs) -> str:
        session_id = kwargs.get("mlflow_session_id", f"sim-{uuid.uuid4().hex[:8]}")
        agent = agents.setdefault(session_id, agent_factory())
        latest_user_msg = input[-1]["content"]
        return chat_turn(agent, latest_user_msg, session_id=session_id)

    return predict_fn


predict_fn_v1 = make_predict_fn(build_agent)


### Run simulation + evaluation in one call

`mlflow.genai.evaluate` accepts the simulator as `data`, runs the simulated conversations turn-by-turn through `predict_fn`, and scores them — all in one step. Because `mlflow.set_active_model(VERSION_V1)` is still in effect, every trace and metric produced here is linked to `VERSION_V1`.


In [ ]:
sim_results_v1 = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=predict_fn_v1,
    scorers=MULTI_TURN_SCORERS,
)

print(f"=== Simulation Evaluation Metrics ({VERSION_V1}) ===")
for metric, value in sim_results_v1.metrics.items():
    print(f"  {metric}: {value}")


In [ ]:
sim_results_v1.result_df


## 9. Compare Agent Versions with the Same Simulator

Version tracking turns the simulator into a regression-testing tool. Below we compare two agent versions using the **same** endpoint, sampling config, test cases, and scorers — the only variable is the **system prompt**. v2 adds instructions that *sound* like an improvement (acknowledge frustration, ask clarifying questions, propose next steps).

| Version | Endpoint | System prompt | Isolates |
|---|---|---|---|
| **v1** | `qwen3-5-0-8b` | base | Baseline |
| **v2** | `qwen3-5-0-8b` | senior-ML-assistant prompt | **Prompt change impact** (v1 vs v2) |

The point of this section is not that v2 is better — it's that we *don't know* until we measure. Small models often mis-apply conditional instructions (for example, acknowledging frustration on every turn, even when the user isn't frustrated). Running both versions through the same simulated conversations and judges reveals whether the prompt change helps or quietly degrades quality.

Each `mlflow.set_active_model(...)` call re-tags downstream traces and evaluation results, so the MLflow UI shows the two versions side by side in the Models view.


In [ ]:
VERSION_V2 = f"{APP_NAME}-v2-prompted-{version_suffix}"

SYSTEM_PROMPT_V2 = (
    "You are a senior ML engineering assistant. When the user is frustrated, "
    "explicitly acknowledge the frustration before answering, ask a concise "
    "clarifying question if the problem is under-specified, and always propose "
    "at least one next-step action the user can try."
)


def build_agent_v2() -> Agent:
    # v2: same endpoint and sampling config as v1, but a more directive
    # system prompt. The v1→v2 delta isolates prompt engineering impact.
    return build_agent(system_prompt=SYSTEM_PROMPT_V2)


# Switch the active model: every subsequent trace & eval result links to v2
mlflow.set_active_model(name=VERSION_V2)
print(f"Active model version: {VERSION_V2}")

# Rebuild the simulator so v2 gets a fresh run against the same test cases
simulator_v2 = ConversationSimulator(
    test_cases=test_cases,
    max_turns=4,
    user_model=JUDGE_MODEL,
)
predict_fn_v2 = make_predict_fn(build_agent_v2)

sim_results_v2 = mlflow.genai.evaluate(
    data=simulator_v2,
    predict_fn=predict_fn_v2,
    scorers=MULTI_TURN_SCORERS,
)

print(f"\n=== Simulation Evaluation Metrics ({VERSION_V2}) ===")
for metric, value in sim_results_v2.metrics.items():
    print(f"  {metric}: {value}")


Side-by-side comparison of the two versions:

**Reading the results**: don't assume the "improved" prompt wins. On a 0.8B model, the v2 prompt's conditional instruction ("*when* the user is frustrated, acknowledge it") is often applied unconditionally — responses to plain factual questions start with "I understand you're frustrated..." and quality drops. If you see v2 scoring at or below v1, that is the regression this workflow is designed to catch *before* shipping the prompt change.


In [ ]:
import pandas as pd

# `results.metrics` only contains scorers whose values can be averaged
# (yes/no judges become a numeric mean). Categorical judges — UserFrustration
# (none/resolved/unresolved) and our tone_consistency judge — carry no mean,
# so they silently disappear from `.metrics`. To compare versions across ALL
# four judges, summarize the per-session values from `result_df` instead.

SCORER_COLS = [
    "conversation_completeness/value",
    "user_frustration/value",
    "knowledge_retention/value",
    "tone_consistency/value",
]


def session_summary(result_df: pd.DataFrame, label: str) -> pd.Series:
    """Distribution of each judge's session-level values for one version."""
    # Session-level assessments live on the first trace of each session,
    # so keep only the rows that actually carry values.
    rows = result_df.dropna(subset=SCORER_COLS, how="all")
    return pd.Series(
        {
            col.replace("/value", ""): ", ".join(
                f"{value}: {count}" for value, count in rows[col].value_counts().items()
            )
            for col in SCORER_COLS
        },
        name=label,
    )


comparison = pd.concat(
    [
        session_summary(sim_results_v1.result_df, VERSION_V1),
        session_summary(sim_results_v2.result_df, VERSION_V2),
    ],
    axis=1,
)
comparison


In the MLflow UI, filter traces by the `LoggedModel` name (`VERSION_V1` or `VERSION_V2`) to inspect each version's runs independently, or open the **Models** tab of the experiment to see both versions side by side with their linked traces, metrics, and evaluation results.


## 10. Single-Turn Side-by-Side Comparison with Ground Truth

Multi-turn evaluation is the main focus of this notebook, but MLflow also supports the classic case of comparing model versions **against a reference answer** on single-turn tasks. This is useful for regression-testing specific knowledge questions before shipping a new agent version — and it gives a second, independent signal on the v1-vs-v2 prompt change from Section 9.

We build an evaluation dataset where each row has:
- `inputs` — what the user asks
- `expectations.expected_response` — the ground-truth reference answer

The questions are deliberately broad: with a small model, overly exacting questions (precise package names, exact API signatures) fail for *every* version and hide the difference between them. Broad questions let the baseline score, so a regression in v2 becomes visible.

Then we run each agent version through the dataset, score with the built-in `Correctness` judge, and display the results side by side.


In [ ]:
from mlflow.genai.scorers import Correctness, Guidelines

# A small ground-truth dataset for regression testing.
# Keep each expected_response to a SINGLE atomic fact: the Correctness judge
# decomposes the expected response into claims and requires the answer to
# support every one of them, so multi-part expectations fail even for
# reasonable answers.
eval_dataset = [
    {
        "inputs": {"question": "What is MLflow experiment tracking used for?"},
        "expectations": {
            "expected_response": "It is used to record and compare machine-learning runs.",
        },
    },
    {
        "inputs": {"question": "What is the relationship between an MLflow experiment and a run?"},
        "expectations": {
            "expected_response": "An experiment groups related runs.",
        },
    },
    {
        "inputs": {"question": "Why would you use an LLM-as-a-judge to evaluate a conversational agent?"},
        "expectations": {
            "expected_response": (
                "Because it can assess subjective response quality automatically and at scale."
            ),
        },
    },
]


### Define a single-turn predict function

The predict function takes an `inputs` dict and returns the agent's reply as a string. We reuse `build_agent` for v1 and `build_agent_v2` for v2. A fresh agent is built per row so there's no cross-row history bleed.


In [ ]:
def make_single_turn_predict_fn(agent_factory):
    def predict_fn(question: str) -> str:
        agent = agent_factory()
        session_id = f"qna-{uuid.uuid4().hex[:8]}"
        return chat_turn(agent, question, session_id=session_id)
    return predict_fn


predict_single_v1 = make_single_turn_predict_fn(build_agent)
predict_single_v2 = make_single_turn_predict_fn(build_agent_v2)


### Evaluate each version against the ground truth

`Correctness` is a built-in LLM-as-a-judge scorer that reads the model's response and compares it to `expectations.expected_response`. It returns `yes`/`no` plus a rationale.


In [ ]:
# V1 run — tagged with VERSION_V1
mlflow.set_active_model(name=VERSION_V1)
gt_results_v1 = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_single_v1,
    scorers=[Correctness(model=JUDGE_MODEL)],
)

# V2 run — tagged with VERSION_V2
mlflow.set_active_model(name=VERSION_V2)
gt_results_v2 = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_single_v2,
    scorers=[Correctness(model=JUDGE_MODEL)],
)

print(f"=== Correctness vs ground truth ===")
print(f"  {VERSION_V1}: {gt_results_v1.metrics}")
print(f"  {VERSION_V2}: {gt_results_v2.metrics}")


### Side-by-side table: question and each version's answer + score

Join the two result dataframes on the question (via their traces) to get a per-question comparison across both versions. This view makes regressions (and rescues) obvious — you can see exactly which questions each version got right or wrong, plus the judge's rationale for each.


In [ ]:
import pandas as pd

# In MLflow 3.11.x, evaluate() returns a result_df with `trace_id` and
# `<scorer>/value` columns only. The question, answer, and judge rationale
# live on the traces, so we join through mlflow.get_trace().


def _trace_details(trace_id: str) -> dict:
    """Extract question, answer, and correctness rationale from a trace."""
    trace = mlflow.get_trace(trace_id)
    root = trace.data.spans[0]
    inputs = root.inputs or {}
    # The root span is chat_turn(agent, user_message, session_id), so the
    # question lives under "user_message" ("question" kept as fallback).
    question = inputs.get("user_message") or inputs.get("question") or str(inputs)
    answer = root.outputs if isinstance(root.outputs, str) else str(root.outputs)
    rationale = ""
    for a in trace.info.assessments or []:
        if a.name == "correctness":
            rationale = a.rationale or ""
    return {"question": question, "answer": answer, "rationale": rationale}


def build_side_by_side(runs: dict[str, "pd.DataFrame"]) -> pd.DataFrame:
    """Join multiple eval result_dfs on the question, via their traces."""
    base = None
    for label, df in runs.items():
        details = df["trace_id"].apply(_trace_details).apply(pd.Series)
        view = pd.concat([details, df["correctness/value"]], axis=1).rename(columns={
            "answer": f"{label}_answer",
            "correctness/value": f"{label}_correct",
            "rationale": f"{label}_rationale",
        })
        base = view if base is None else base.merge(view, on="question", how="outer")
    return base


side_by_side = build_side_by_side({
    VERSION_V1: gt_results_v1.result_df,
    VERSION_V2: gt_results_v2.result_df,
})
side_by_side


Click any row in the MLflow UI (Experiment → Evaluation → Results) to open the full trace for both versions. Each row is individually traceable, so you can drill into token usage, latency, and tool calls per question and per version — a view DeepEval and similar single-turn-only evaluators don't provide out of the box.

### When to use single-turn vs multi-turn evaluation

| Question | Use |
|---|---|
| Does the agent answer this *specific knowledge question* correctly? | Single-turn with ground truth (this section) |
| Does the agent handle *this type of conversation* without dropping context or frustrating the user? | Multi-turn with session-level scorers (sections 6–9) |
| Does a new agent version regress on a reference Q&A set? | Single-turn side-by-side comparison (this section) |
| Does a new agent version regress on conversation flow? | Multi-turn simulation with `ConversationSimulator` (sections 8–9) |

Combining both gives you a regression safety net across the full spectrum of agent quality signals.


## 11. Best Practices for Conversational AI Quality Assurance

### Evaluation strategy by stage

| Stage | Approach | Scorers |
|---|---|---|
| **Development** | Simulate conversations with diverse personas | `ConversationCompleteness`, `UserFrustration`, custom judges |
| **Pre-release** | Replay production-derived test cases against new versions | `KnowledgeRetention`, `ConversationalRoleAdherence` |
| **Production** | Evaluate live sessions continuously | `UserFrustration`, `ConversationalSafety` |

### Recommendations

1. **Tag every trace with a session ID** — use `mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})` so MLflow can group turns
2. **Version-track your agent** — call `mlflow.set_active_model(name=version_name)` before any traces are produced, so every trace and evaluation result links to a specific agent version
3. **Combine single-turn and multi-turn scorers** — single-turn catches per-response issues (safety, relevance); multi-turn catches session-level problems (frustration, completeness)
4. **Use `simulation_guidelines`** to reproduce specific failure patterns observed in production
5. **Extract test cases from production** with `generate_test_cases()` to build regression tests from real conversations
6. **Version your test scenarios** as MLflow Evaluation Datasets for reproducible comparisons across agent versions
7. **Run simulations in CI/CD** to catch regressions before deployment


## Cleanup

Delete the SageMaker AI endpoint and, if no longer needed, the MLflow App:

```python
import boto3
sm = boto3.client("sagemaker", region_name=REGION)
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
sm.delete_model(ModelName=ENDPOINT_NAME)
```

The MLflow App itself is serverless and at no additional cost, but it stores traces and artifacts in Amazon S3 which continue to incur storage charges.

## Next Steps

- **View results** in the MLflow UI → experiment `multi-turn-eval-demo` → Sessions tab and Models tab
- **Add more built-in judges**: `ConversationalGuidelines`, `ConversationalRoleAdherence`, `ConversationalSafety`, `ConversationalToolCallEfficiency`
- **Extract test cases from production**: `mlflow.genai.simulators.generate_test_cases(sessions)` to build regression scenarios from real conversations
- **Persist test cases**: save to MLflow Evaluation Datasets with `create_dataset()` for reproducible testing
- **Automate evaluation**: configure MLflow to run judges automatically on new traces as they're logged
- **Dive deeper into version tracking**: see the [MLflow Version Tracking guide](https://mlflow.org/docs/latest/genai/version-tracking/) for external code management, deployment patterns, and more systematic v1-vs-v2 comparison workflows
